#### Transform Orders Data-Strings to JSON 
- Pre process the json string to fix data quality issues
- Transform JSON string to JSON object
- Write transformed data to silver layer

#### 1. Pre process the json string to fix data quality issues
- https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_replace

In [0]:
%sql
SELECT
    value,
    regexp_replace(
        value, 
        '"order_date": (\\d{4}-\\d{2}-\\d{2})', 
        '"order_date": "\$1"'
    ) AS value_new
FROM gizmobox.bronze.v_orders

In [0]:
%sql
WITH cleaned_json AS
(
SELECT
    value,
    regexp_replace(
        value, 
        '"order_date": (\\d{4}-\\d{2}-\\d{2})', 
        '"order_date": "\$1"'
    ) AS value_new
FROM gizmobox.bronze.v_orders
)
SELECT 
    value_new,
    value:order_id,
    value_new:order_id 
FROM cleaned_json

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW tv_orders_fixed AS
SELECT
    regexp_replace(
        value, 
        '"order_date": (\\d{4}-\\d{2}-\\d{2})', 
        '"order_date": "\$1"'
    ) AS value_new
FROM gizmobox.bronze.v_orders

####2. Transform JSON string to JSON object
- Function https://docs.databricks.com/aws/en/sql/language-manual/functions/schema_of_json
- Function https://docs.databricks.com/aws/en/sql/language-manual/functions/from_json

In [0]:
%sql
SELECT
    schema_of_json(value_new) AS schema,
    value_new
FROM tv_orders_fixed
LIMIT 1;

In [0]:
%sql
SELECT
    from_json( 
        value_new, 
        'STRUCT<customer_id: BIGINT, 
        items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT,quantity: BIGINT>>, 
        order_date: STRING, 
        order_id: BIGINT, 
        order_status: STRING, 
        payment_method: STRING, 
        total_amount: BIGINT, 
        transaction_timestamp: STRING>') AS json_string
FROM tv_orders_fixed


####3. Write transformed data to silver layer

In [0]:
%sql
CREATE TABLE gizmobox.silver.orders_json
AS 
SELECT
    from_json( 
        value_new, 
        'STRUCT<customer_id: BIGINT, 
        items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT,quantity: BIGINT>>, 
        order_date: STRING, 
        order_id: BIGINT, 
        order_status: STRING, 
        payment_method: STRING, 
        total_amount: BIGINT, 
        transaction_timestamp: STRING>') AS json_string
FROM tv_orders_fixed

In [0]:
%sql
SELECT * from gizmobox.silver.orders_json